# reduce-op-mean-divide — faded example 2: Harmonic mean: reciprocal then sum-divide-invert

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `reduce-op-mean-divide`. Running the beacon reports progress on the `Distributed: reduce-op mean divide` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: reduce-op mean divide` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`reduce-op-mean-divide`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "reduce-op-mean-divide"
DD_SUBTOPIC = "Distributed: reduce-op mean divide"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The harmonic mean is `N / sum(1/x_r)`. Operationally: transform each rank's value to its reciprocal, `all_reduce(SUM)`, divide by `world_size` to mean the reciprocals, then invert. The reciprocal transform is the step that makes a SUM reduction compute the right thing.

## Faded exercise 2

Complete `harmonic_mean`. The all_reduce, divide, and final inversion are filled. Fill in the construction of the reciprocal tensor that is reduced.

**Fill in:** build the length-1 tensor holding the reciprocal of local_value

In [ ]:
class FakeDist:
    def __init__(self, recip_contribs):
        self.total = sum(recip_contribs)
    def all_reduce(self, tensor, op='sum'):
        tensor.copy_(t.tensor([self.total]))


def harmonic_mean(world_size, local_value, dist_module):
    if local_value <= 0:
        raise ValueError('undefined for non-positive')
    tensor = None  # TODO: build the length-1 tensor holding the reciprocal of local_value
    dist_module.all_reduce(tensor, op='sum')
    tensor /= world_size
    return 1.0 / tensor.item()


values = [100.0, 50.0]
fd = FakeDist([1.0 / v for v in values])
result = harmonic_mean(len(values), 100.0, fd)

def _test():
    values = [100.0, 50.0]
    fd = FakeDist([1.0 / v for v in values])
    got = harmonic_mean(len(values), 100.0, fd)
    # H = 2 / (1/100 + 1/50) = 2 / 0.03 = 66.666...
    expected = len(values) / sum(1.0 / v for v in values)
    assert abs(got - expected) < 1e-4, f'expected {expected}, got {got}'
    assert abs(got - 66.6667) < 1e-2

try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
class FakeDist:
    def __init__(self, recip_contribs):
        self.total = sum(recip_contribs)
    def all_reduce(self, tensor, op='sum'):
        tensor.copy_(t.tensor([self.total]))


def harmonic_mean(world_size, local_value, dist_module):
    if local_value <= 0:
        raise ValueError('undefined for non-positive')
    tensor = t.tensor([1.0 / local_value])
    dist_module.all_reduce(tensor, op='sum')
    tensor /= world_size
    return 1.0 / tensor.item()


values = [100.0, 50.0]
fd = FakeDist([1.0 / v for v in values])
result = harmonic_mean(len(values), 100.0, fd)
```
</details>